In [2]:
pip install gspread google-auth pandas

In [1]:
# import pandas as pd


# df_semanal = pd.read_csv( "../Datos_csv/df_semanal.csv",index_col=0,parse_dates=True)
# df_semanal_1 = pd.read_csv("../Datos_csv/df_semanal_1.csv",index_col=0,parse_dates=True)
# df_semanal_2 = pd.read_csv("../Datos_csv/df_semanal_2.csv",index_col=0,parse_dates=True)
# df_semanal_3 = pd.read_csv("../Datos_csv/df_semanal_3.csv",index_col=0,parse_dates=True)
# df_semanal_4 = pd.read_csv("../Datos_csv/df_semanal_4.csv",index_col=0,parse_dates=True)
# df_semanal_5 = pd.read_csv("../Datos_csv/df_semanal_5.csv",index_col=0,parse_dates=True)
# df_semanal_6 = pd.read_csv("../Datos_csv/df_semanal_6.csv",index_col=0,parse_dates=True)
# df_semanal_7 = pd.read_csv("../Datos_csv/df_semanal_7.csv",index_col=0,parse_dates=True)
# df_semanal_8 = pd.read_csv("../Datos_csv/df_semanal_8.csv",index_col=0,parse_dates=True)

In [4]:
from datetime import datetime

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials

# CONFIG

SERVICE_ACCOUNT_FILE = "credenciales_google.json"

SHEET_URL = "https://docs.google.com/spreadsheets/d/1Rhh-1eRsS827P1EMzDH6hjIeNkHoTb-jmbhDh9fkR8I/edit?gid=661715534#gid=661715534"

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]


# CONEXIÓN
def conectar_sheet():
    creds = Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=SCOPES
    )
    client = gspread.authorize(creds)
    spreadsheet = client.open_by_url(SHEET_URL)

 
    worksheet = spreadsheet.get_worksheet(0)
    return worksheet


# LEER DATOS

def leer_experimentos(worksheet):
    records = worksheet.get_all_records()
    df = pd.DataFrame(records)
    return df


# ENCONTRAR PRIMER PENDIENTE

def buscar_primera_fila_pendiente(df):
    for idx, row in df.iterrows():
        estado = str(row.get("Estado", "")).strip().upper()
        if estado == "PENDIENTE":
            # idx es índice pandas empezando en 0
            # en sheets:
            # fila 1 = cabecera
            # fila 2 = primer registro
            fila_sheet = idx + 2
            return fila_sheet, row.to_dict()
    return None, None


# ACTUALIZAR FILA A RUNNING

def marcar_como_running(worksheet, fila_sheet):
    fecha_ini = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Columnas:
    # A Archivo
    # B Target
    # C Método
    # D FechaIni
    # E FechaFin
    # F RMSE
    # G Baseline
    # H Estado

    worksheet.update(
        f"D{fila_sheet}:H{fila_sheet}",
        [[fecha_ini, "", "", "", "RUNNING"]]
    )

    return fecha_ini


# ACTUALIZAR FILA A DONE 

def marcar_como_done(worksheet, fila_sheet, rmse, baseline):
    fecha_fin = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    worksheet.update(
        f"E{fila_sheet}:H{fila_sheet}",
        [[fecha_fin, rmse, baseline, "DONE"]]
    )

    return fecha_fin


# MAIN
def main():
    worksheet = conectar_sheet()

    df = leer_experimentos(worksheet)
    print("Primeras filas leídas:")
    print(df.head())

    fila_sheet, experimento = buscar_primera_fila_pendiente(df)

    if fila_sheet is None:
        print("No hay experimentos pendientes.")
        return

    print("\nPrimera fila pendiente encontrada:")
    print("Fila en Google Sheets:", fila_sheet)
    print(experimento)

    fecha_ini = marcar_como_running(worksheet, fila_sheet)
    print(f"\nFila {fila_sheet} marcada como RUNNING en FechaIni={fecha_ini}")

    # -----------------------------------------------------
    # SOLO PARA PRUEBA:
    # descomenta esto si quieres dejarla terminada
    # -----------------------------------------------------
    fecha_fin = marcar_como_done(worksheet, fila_sheet, rmse=0.1234, baseline=0.1567)
    print(f"Fila {fila_sheet} marcada como DONE en FechaFin={fecha_fin}")


if __name__ == "__main__":
    main()

Primeras filas leídas:
            Archivo      Target            Método FechaIni FechaFin RMSE  \
0  df_semanal_1.csv  target_BND  LinearRegression                          
1  df_semanal_1.csv  target_AGG  LinearRegression                          
2  df_semanal_1.csv  target_DBC  LinearRegression                          
3  df_semanal_1.csv  target_DIA  LinearRegression                          
4  df_semanal_1.csv  target_DVY  LinearRegression                          

  Baseline     Estado  
0           PENDIENTE  
1           PENDIENTE  
2           PENDIENTE  
3           PENDIENTE  
4           PENDIENTE  

Primera fila pendiente encontrada:
Fila en Google Sheets: 2
{'Archivo': 'df_semanal_1.csv', 'Target': 'target_BND', 'Método': 'LinearRegression', 'FechaIni': '', 'FechaFin': '', 'RMSE': '', 'Baseline': '', 'Estado': 'PENDIENTE'}


C:\Users\aleja\AppData\Local\Temp\ipykernel_19916\2665786783.py:71: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update(



Fila 2 marcada como RUNNING en FechaIni=2026-03-31 21:25:37


C:\Users\aleja\AppData\Local\Temp\ipykernel_19916\2665786783.py:84: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update(


Fila 2 marcada como DONE en FechaFin=2026-03-31 21:25:38
